# Cell 1: Problem Statement and Imports

## Lending Club Loan Data Analysis
**Objective:** Build a deep learning model to predict the chance of loan default using historical data from 2007 to 2015.

### Dataset Definition:
- **credit.policy**: 1 if criteria met, 0 otherwise.
- **purpose**: The category of the loan.
- **fico**: The FICO credit score of the borrower.
- **not.fully.paid**: The target variable (1 = default, 0 = paid).

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

ImportError: DLL load failed while importing _tools: An Application Control policy has blocked this file.

# Cell 2: Load and Inspect Data

In [ ]:
# Load the dataset
df = pd.read_csv('./data/loan_data.csv')

# Initial inspection
print(df.info())
df.head()

## Cell 3: Task 1 - Feature Transformation

### 1. Feature Transformation
We transform the categorical column 'purpose' into numerical values using One-Hot Encoding.
We use `drop_first=True` to avoid the dummy variable trap.

In [ ]:
final_data = pd.get_dummies(df, columns=['purpose'], drop_first=True)
final_data.head()


## Cell 4: Task 2 - Exploratory Data Analysis (EDA)

### 1. Feature Transformation
We transform the categorical column 'purpose' into numerical values using One-Hot Encoding.
We use `drop_first=True` to avoid the dummy variable trap.

In [ ]:

plt.figure(figsize=(10,6))
df[df['not.fully.paid']==1]['fico'].hist(alpha=0.5, color='blue', bins=30, label='not.fully.paid=1')
df[df['not.fully.paid']==0]['fico'].hist(alpha=0.5, color='red', bins=30, label='not.fully.paid=0')
plt.legend()
plt.xlabel('FICO')
plt.title('FICO Distribution by Payment Status')
plt.show()

plt.figure(figsize=(11,7))
sns.countplot(x='purpose', hue='not.fully.paid', data=df, palette='Set1')
plt.title('Loan Purpose Count by Payment Status')
plt.show()

## Cell 5: Task 3 - Additional Feature Engineering (Correlation)

### 3. Additional Feature Engineering
Checking for high correlation between features. We define a threshold of 0.8 to drop redundant features.

In [ ]:

plt.figure(figsize=(12,10))
sns.heatmap(final_data.corr(), cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Logic to identify features with correlation higher than 0.8
corr_matrix = final_data.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]

print(f"Features to drop: {to_drop}")
# Drop if any found (usually empty for this dataset)
final_data.drop(to_drop, axis=1, inplace=True)

## Cell 6: Data Preprocessing (Split and Scale)

In [ ]:
# Define Features and Target
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

# Split data (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=101)

# Normalize the data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Cell 7: Task 4 - Modeling

### 4. Modeling
Building a Deep Learning model using Keras.

In [ ]:

model = Sequential()

# Input layer & Hidden Layers
model.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.2))

model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))

# Output layer (Sigmoid for binary classification)
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train,
                    epochs=25,
                    batch_size=256,
                    validation_data=(X_test, y_test),
                    verbose=1)

## Cell 8: Evaluation

### 5. Evaluation
Reviewing the confusion matrix and classification report.

In [ ]:

predictions = (model.predict(X_test) > 0.5).astype("int32")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

print("\nClassification Report:")
print(classification_report(y_test, predictions, zero_division=0))